# Calibrate a real robot from telemetry

Every other tutorial roots its controller, its binary, and its attestation on a
**calibrated robot** — the single source of truth. This one shows where that
calibrated robot *comes from*: you excite the physical unit, measure how it moves,
and fit the parameters that make the model match. The result is a robot whose
identity (and attestation handle) reflects *this* unit, not a datasheet.

The loop:

```
 design a safe excitation ─► validate (limit-bounded) ─► run on the bus ─► fit (LM)
        │                                                                     │
        └──────────────── the same forward-mode fit, hardware or sim ─────────┘
                                                                              ▼
                                                              SysidResult ─► calibrated Robot
```

We'll recover known parameters in simulation (so we can check the fit is exact),
show the **split capture/fit** that lets a Raspberry Pi capture while a workstation
fits, and close with the honest lesson from running it on a real LeKiwi.

In [1]:
import logging, warnings
for _n in ("jaxonomy", "jaxterity", "mujoco", "absl"):
    logging.getLogger(_n).setLevel(logging.ERROR)
logging.disable(logging.WARNING)   # quiet the per-simulate "Simulator ready" INFO lines
warnings.filterwarnings("ignore")
import numpy as np, jax.numpy as jnp
import jaxterity.zoo as zoo
from jaxterity.calibrate.active import (active_identify, LoopbackLink, ReplayLink,
    design_coast_excitation, validate_excitation, dof_joint_names)
from jaxterity.errors import CalibrationError
robot = zoo.load("so100")   # the SO-101 / LeKiwi arm
print("robot:", robot.name, "| joints:", dof_joint_names(robot))

robot: so101_new_calib | joints: ('shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper')


## Step 1 — Design a *safe* excitation

The probe is a gentle **coast**: each joint gets a small initial velocity and the
arm coasts under its own dynamics — the regime that reveals damping/friction.
Before anything reaches the bus, `validate_excitation` simulates it and **rejects**
any profile that would drive a joint past a position or velocity limit. Safety is
checked in software first.

In [2]:
exc = design_coast_excitation(robot, speed=0.1)   # 0.1 rad/s kick on every joint
validate_excitation(robot, exc)                    # passes (no raise)
print("designed coast probe: qd0 =", [round(v,2) for v in exc.qd0], "rad/s,",
      "sampled at", exc.sample_times, "s -> within limits ✓")

reckless = design_coast_excitation(robot, speed=50.0)   # absurdly fast
try:
    validate_excitation(robot, reckless)
    print("reckless probe accepted?!")
except CalibrationError as e:
    print("\nreckless probe (50 rad/s) -> REJECTED before any command is sent:")
    print("  ", str(e)[:110], "…")

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


designed coast probe: qd0 = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1] rad/s, sampled at (0.1, 0.2) s -> within limits ✓

reckless probe (50 rad/s) -> REJECTED before any command is sent:
   active sysid excitation would exceed the velocity limit of joint 'shoulder_pan': |qd0|=50 rad/s > 10 rad/s. Fi …


## Step 2 — Recover known parameters (in simulation)

`LoopbackLink` is a *seeded simulated arm*: it runs the excitation through the
robot's own dynamics with **planted** parameters. The fit must recover them — and
because it's deterministic, a fixed seed gives a bit-identical result. This is how
the method is validated without touching hardware.

In [3]:
TRUE = 0.12
targets = ("shoulder_lift.damping", "elbow_flex.damping")
link = LoopbackLink(robot, {t: TRUE for t in targets})
result = active_identify(robot, link, targets, excitation=exc)
print("planted damping:", TRUE)
for t in targets:
    print(f"  recovered {t:24s} = {float(result.fitted[t]):.6f}")
print("\nfinal-state RMS residual:", f"{result.residuals['final_state_rms']:.2e}",
      "| recipe:", result.recipe_name)

planted damping: 0.12
  recovered shoulder_lift.damping    = 0.120000
  recovered elbow_flex.damping       = 0.120000

final-state RMS residual: 1.80e-07 | recipe: active.sysid


## Step 3 — Apply the fit → the robot's identity moves

A `SysidResult` isn't just numbers — `apply_to` produces a **calibrated robot**
with the fitted parameters *and* a provenance record (the recipe + the telemetry
hash). Its attestation handle changes, so everything built from it downstream
(controller, binary, manifest — tutorials #1/#4) now tracks this calibration.

In [4]:
calibrated = result.apply_to(robot)
print("handle before:", robot.attestation_handle[:24], "…")
print("handle after :", calibrated.attestation_handle[:24], "…")
print("handle moved :", robot.attestation_handle != calibrated.attestation_handle)
print("provenance   :", [r[0] for r in calibrated.provenance])

handle before: 0c7980bf0b27fb2e8b5cfb3c …
handle after : c443737ab84cf190807f99bb …
handle moved : True
provenance   : ['active.sysid']


## Step 4 — Split capture/fit: the Pi captures, the workstation fits

On real hardware the servo bus lives on a small computer (a Raspberry Pi), but the
**fit** needs the full JAX/MJX simulation stack. `ReplayLink` bridges them: the
device streams the excitation and records `(q, qd)` telemetry; the workstation
replays that telemetry through the **same** `active_identify`. A telemetry-file
handoff, not a fit handoff — and the result is *identical* to running it live.

In [5]:
# the "device" captures telemetry over the bus (here, the loopback stands in)...
captured = link.run_excitation(exc)
print("captured telemetry:", captured.q.shape[0], "samples x", captured.q.shape[1], "joints")

# ...the "workstation" replays it through the identical fit
replay_result = active_identify(robot, ReplayLink(captured), targets, excitation=exc)
same = all(float(replay_result.fitted[t]) == float(result.fitted[t]) for t in targets)
print("replayed fit == live fit (bit-for-bit):", same)
print("-> capture on the Pi, fit on the workstation; nothing about the result changes.")

captured telemetry: 2 samples x 6 joints


replayed fit == live fit (bit-for-bit): True
-> capture on the Pi, fit on the workstation; nothing about the result changes.


## Step 5 — On a real LeKiwi: an honest lesson

Run this on a physical LeKiwi and you learn something the simulation can't tell
you. We validated the bus + capture + `ReplayLink` path on a real unit (all 9
STS3215 servos), then ran a single-joint free coast on `shoulder_pan` (the one
gravity-neutral arm joint that can safely torque-off coast):

> the joint decelerated from ~0.54 rad/s **to rest in ~170 ms** (exponential
> τ ≈ 79 ms, effective damping ≈ 12.7 s⁻¹).

That fast stop *is* the finding: a **geared Feetech servo doesn't free-coast** like
the frictionless model — gear friction dominates, and the gravity-loaded joints
can't safely torque-off coast at all. So the free-coast observer is a poor match
for a real servo-in-the-loop coast. The bus, the capture, and the `ReplayLink`
fit-seam are all validated; the limitation is **physical**, not in the code — the
kind of truth you only get by putting the method on real hardware.

## Recap

You produced a **calibrated robot from measured motion**:

- a **limit-bounded coast probe**, rejected in software if it would exceed a limit;
- a **forward-mode fit** that recovers planted parameters exactly in simulation;
- `apply_to` → a calibrated robot whose **attestation handle moves**, re-rooting
  every downstream artifact on this unit;
- a **split capture/fit** (`ReplayLink`) so the Pi captures and the workstation
  fits — bit-for-bit identical;
- and an honest hardware lesson about where the free-coast model meets a real
  geared servo.

This calibrated robot is exactly the source of truth tutorials #1 and #4 build on.
Next: **the predictive self-check co-processor** — run a per-unit twin beside the
controller to catch overheating and brownout before they happen.